Cross-checking against milieu_anteil, however, shows the raw resemblance ranking is not yet an action list: several top areas (Uferstraße 0.49, Großgörschenstraße 0.50) are already near the 50% majority threshold, so their high score partly reflects existing protection rather than an unprotected look-alike. This motivates a second layer that discounts resemblance by how much protection is already in place.

## Creating a watchlist, filtered by urgency 

Because resemblance alone rewards areas that are already partly protected, an urgency score is introduced: oof_prob × (1 − milieu_anteil), combining "looks designated" with "still unprotected" into one priority. 

This is explicitly a filtering device, not a model prediction. The variable "milieu_anteil" never enters the model as a feature and is used here only to reweight the ranking. The continuous share is drawn from the target notebook, where a value of zero is an observed "no protection", not a missing value.

In [ ]:
#Refine: milieu_anteil > 0 is too coarse. Split by how much protection already exists
def classify(a):
    if a < 0.10:
        return "effectively unprotected"      # early-warning candidate
    elif a < 0.40:
        return "partially protected"          # protection started, substantial gap
    else:
        return "near-designated"              # just below the 50% majority threshold

resemblance_checked["status"] = resemblance_checked["milieu_anteil"].apply(classify)
print(resemblance_checked["status"].value_counts().to_string(), "\n")
print(resemblance_checked[["plr_name", "bez", "oof_prob", "milieu_anteil", "status"]].to_string())

In [ ]:
#check: among areas the model flags, where is the early-warning value highest?

labels_full = pd.read_csv("../../data/Module 2/target_milieuschutz.csv", dtype={"plr_id": str})
labels_full["plr_id"] = labels_full["plr_id"].astype(str).str.zfill(8)

#pull milieu_anteil onto the list
resemblance_checked = resemblance.merge(
    labels_full[["plr_id", "milieu_anteil"]], on="plr_id", how="left"
)
resemblance_checked.index = resemblance_checked.index + 1
resemblance_checked.index.name = "rank"


print(resemblance_checked[["plr_name", "bez", "oof_prob", "milieu_anteil"]].to_string())

## Result 

Adding the variable milieu_anteil lets us see that some of the very high lookalikes are nearly 50% protected. This is why we create a variable that states clearly whether they are protected, partially protected or not. 

## B Mapping the Watchlist (filtered by urgency)


In [ ]:
#labels_full holds milieu_anteil (from target_milieuschutz.csv) — kept out of df on purpose
plot_geo2 = (
    gdf
    .merge(df[["plr_id", "oof_prob", "y_true", "plr_name"]], on="plr_id", how="left")
    .merge(labels_full[["plr_id", "milieu_anteil"]], on="plr_id", how="left")
)

#Plot 2: urgency only for UNDESIGNATED PLR; designated + non-modelled stay NaN (grey)
undes2 = (plot_geo2["y_true"] == 0)
plot_geo2["urgency"] = np.nan
plot_geo2.loc[undes2, "urgency"] = (
    plot_geo2.loc[undes2, "oof_prob"] * (1 - plot_geo2.loc[undes2, "milieu_anteil"])
)

#diagnostics: how many geometries got no model row (the ~15 dropped PLR)?
n_nomatch2 = plot_geo2["oof_prob"].isna().sum()
print(f"{len(plot_geo2)} geometries | {n_nomatch2} without a model row (dropped/uninhabited PLR)")
print(f"urgency range: {plot_geo2['urgency'].min():.3f} – {plot_geo2['urgency'].max():.3f}")

In [ ]:
#plot2: urgency as model similarity x unprotected share 
#where is the early-warning value highest" — high similarity and little existing protection
fig, ax = plt.subplots(figsize=(12, 11))
red_cmap = LinearSegmentedColormap.from_list("kiez_red", ["#FFF5F0", "#EE4B2B", "#8B0000"])


#layer 1: all PLR as light grey base (so dropped + designated areas are visible, not holes)
plot_geo2.plot(ax=ax, color="#e8e8e8", edgecolor="white", linewidth=0.3)

#layer 2: designated PLR in a distinct neutral (already protected — not "urgent")
plot_geo2[plot_geo2["y_true"] == 1].plot(
    ax=ax, color="#b0b8c0", edgecolor="white", linewidth=0.3
)

#layer 3: undesignated PLR coloured by urgency (white -> red)
plot_geo2[undes2].plot(
    ax=ax, column="urgency", cmap=red_cmap, edgecolor="white", linewidth=0.3,
    legend=True, legend_kwds={"label": "Urgency  (model similarity × unprotected share)",
                              "shrink": 0.5}
)

#Bezirk outlines so the weak-fold districts are locatable
gdf.dissolve("bez").boundary.plot(ax=ax, color="#444444", linewidth=0.7)

ax.set_title("Urgency Map - model similarity x unprotected share\n",
             fontsize=13, fontweight="bold", loc="left")
ax.axis("off")
plt.tight_layout()
plt.savefig("../../plots/Module2/M2_watchplot.png", dpi=200, bbox_inches="tight")
plt.show()

## Result 

The urgency layer reweights the resemblance score by the unprotected share, so areas that merely sit near the 50% threshold recede and genuinely unprotected look-alikes rise. 

The candidates cluster in the inner-city ring directly around the existing grey (already-protected) areas — Kreuzberg, Neukölln, Friedrichshain and the corridor toward the south-east — rather than at the city edge. That spatial pattern is exactly what the early-warning thesis predicts and serves as a plausibility check on the map.

## Comparison of the two Lists 

The two rankings — raw resemblance (oof_prob) and protection-discounted urgency — agree on most areas but diverge on seven: seven PLR that the resemblance list includes are pushed out of the urgency top-25, and seven different ones enter. 

The areas leaving are the near-designated cases (milieu_anteil around 0.4–0.5) whose scores the discount deliberately shrinks. The areas entering are lower-resemblance but essentially unprotected. This ~7-of-25 shift is the practical footprint of the urgency reweighting.

In [ ]:
#where are the two lists differing? 
sub = plot_geo2[undes2].copy()
sub["rank_prob"] = sub["oof_prob"].rank(ascending=False)
sub["rank_urg"]  = sub["urgency"].rank(ascending=False)

wl = sub.nsmallest(25, "rank_prob")          
print(wl[["oof_prob", "milieu_anteil", "urgency"]]
      .assign(shift=wl["rank_prob"] - wl["rank_urg"])
      .sort_values("shift", key=abs, ascending=False)
      .head(10))

#does urgency pull anyone INTO the top 25 who wasn't there by oof_prob?
in_prob = set(sub.nsmallest(25, "rank_prob").index)
in_urg  = set(sub.nsmallest(25, "rank_urg").index)
entrants = sorted(in_urg - in_prob)          
print("only in oof top25:", sorted(in_prob - in_urg))
print("only in urgency top25:", entrants)

## Result

7 PLRs switch sides, which ones come in in the urgency watchlist? Do they also come from the weak fold (Charlottenburg-Wilmersdorf and Lichtenberg)? 

In [ ]:
#build sub FIRST
sub = plot_geo2[undes2].copy()

#strip the "NN - " prefix
sub["bez_name"] = sub["bez"].str.split(" - ").str[-1].str.strip()

#define what counts as weak fold
weak_fold_names = ["Charlottenburg-Wilmersdorf", "Lichtenberg"]

#pick the entrants and include bez_name in the columns
cols = ["bez", "bez_name", "oof_prob", "milieu_anteil", "urgency"]
entrant_rows = sub.loc[entrants, cols].copy()

#match on bez_name (stripped), NOT bez (prefixed)
entrant_rows["weak_fold"] = entrant_rows["bez_name"].isin(weak_fold_names)

entrant_rows = entrant_rows.sort_values("oof_prob", ascending=False)
print("only in urgency top25:", entrants)
print(entrant_rows)
print(f"\n{entrant_rows['weak_fold'].sum()} of {len(entrant_rows)} entrants sit in the weak fold")

### Results - to be rewritten

The urgency watchlist is the union of both top-25 lists — 32 areas in total — with provenance flags recording whether each entered via resemblance, urgency, or both. Of the seven urgency entrants, only one falls in the weak Charlottenburg-Wilmersdorf/Lichtenberg fold; it is retained rather than excluded, since the weak fold is a data property, not a disqualifier, though its oof_prob (high-0.60s) sits at the lower end of the list and should be read with that caveat. 

The candidates lie almost entirely inner-city, adjacent to already-protected areas — a strong sanity signal that the map is plausible. Because the urgency surface looks so convincing, it bears repeating that it is a derived interpretive score (resemblance × unprotected share), not a direct model output. Grey areas are either already designated or not modelled.

## to be rewritten, the urgency list, not the watchlist anymore.

In [ ]:
#Export the WATCHLIST: top-25 undesignated PLR by urgency (oof_prob x unprotected share)
#This is the policy-facing shortlist (resemblance filtered by how much protection already exists).

N_WATCHLIST = 25

und = plot_geo2[undes2].copy()
und = und.merge(df[["plr_id", "plr_name"]], on="plr_id", how="left")
und["urgency"] = und["oof_prob"] * (1 - und["milieu_anteil"])
und["rank_urgency"] = und["urgency"].rank(ascending=False, method="min").astype(int)
und["status"] = und["milieu_anteil"].apply(classify)

watchlist = (
    und[und["rank_urgency"] <= N_WATCHLIST]
    .sort_values("rank_urgency")
    [["plr_id", "plr_name", "bez", "oof_prob", "urgency", "milieu_anteil", "status", "rank_urgency"]]
    .reset_index(drop=True)
)

watchlist.to_csv("../../models/M2_watchlist.csv", index=False)
print(f"exported {len(watchlist)} PLR (urgency top-{N_WATCHLIST}) -> M2_watchlist.csv")
print(watchlist.to_string(index=False))

### Caveats — and how we validate against them

Two caveats apply to the urgency score. First, oof_prob and milieu_anteil are correlated by construction: the model is trained to predict designation (milieu_anteil > 0.5), so a PLR at 0.49 scores high precisely because it sits near the designation pattern, and urgency then multiplies that high score by a small (1 − 0.49). 

Under a filtering framing this is not a flaw but the intent — deliberately combining "looks designated" with "still unprotected" into one priority — but the two factors should not be read as independent signals confirming each other. Second, the 0.50 threshold is a hard gate: an area at 0.49 is scored, at 0.51 it drops off entirely; this is a defensible scope decision (majority-protected areas are outside the action space), and the (1 − milieu_anteil) weighting further encodes a contestable policy judgment — that urgency falls linearly with existing protection.

The practical stance is therefore to keep oof_prob as the primary, statistically grounded ranking and treat urgency as an explicitly labelled interpretive filter, not a model output. 

Next, we will cross-check and validate our findings with two independent checks: the library cleanlab in Notebook 4 will audit the labels themselves. A regression cross-check in notebook 5 will re-derive the ranking from the continuous milieu_anteil instead of the binary label.